# Figure 6: Gene-module compactness

Run after the training and evaluation commands in `bash/paper/`. SCENE outputs use the `scLDM` names referenced below.


In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment_scene.yaml").exists())
os.chdir(PROJECT_ROOT / "notebooks" / "figures")
for folder in ("fig_1", "fig_2", "fig_3", "fig_4", "fig_5", "fig_6", "app", "qc"):
    Path("output", folder).mkdir(parents=True, exist_ok=True)


In [ ]:
import math
from functools import reduce

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.cm import ScalarMappable

# -------------------------
# Figure dimensions
# -------------------------
MM_TO_INCH = 1 / 25.4

# Final figure width at journal size
# Use 180 mm for double-column, 90 mm for single-column
FIG_W_MM = 160
FIG_H_MM = 50

FIG_W = FIG_W_MM * MM_TO_INCH
FIG_H = FIG_H_MM * MM_TO_INCH

# Font sizes
BASE_FONTSIZE = 5.5
TITLE_FONTSIZE = 6.5
LABEL_FONTSIZE = 6
TICK_FONTSIZE = 5
ANNOT_FONTSIZE = 5
CBAR_LABEL_FONTSIZE = 6
CBAR_TICK_FONTSIZE = 5

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": BASE_FONTSIZE,
    "axes.titlesize": TITLE_FONTSIZE,
    "axes.labelsize": LABEL_FONTSIZE,
    "xtick.labelsize": TICK_FONTSIZE,
    "ytick.labelsize": TICK_FONTSIZE,
    "pdf.fonttype": 42,   # editable text in Illustrator
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

sns.set_style("white")

# -------------------------
# Input datasets
# -------------------------
datasets = {
    "SIMBA": {
        "Control": "../../results/geometric_biology/SIMBA_GR_00/gene_module_eval.csv",
        "GR 1h": "../../results/geometric_biology/SIMBA_GR_01/gene_module_eval.csv",
        "GR 2h": "../../results/geometric_biology/SIMBA_GR_02/gene_module_eval.csv",
        "GR 4h": "../../results/geometric_biology/SIMBA_GR_04/gene_module_eval.csv",
        "GR 8h": "../../results/geometric_biology/SIMBA_GR_08/gene_module_eval.csv",
        "GR 18h": "../../results/geometric_biology/SIMBA_GR_18/gene_module_eval.csv",
    },
    "scLDM 3D": {
        "Control": "../../results/geometric_biology/scLDM_GR_00_3D/gene_module_eval.csv",
        "GR 1h": "../../results/geometric_biology/scLDM_GR_01_3D/gene_module_eval.csv",
        "GR 2h": "../../results/geometric_biology/scLDM_GR_02_3D/gene_module_eval.csv",
        "GR 4h": "../../results/geometric_biology/scLDM_GR_04_3D/gene_module_eval.csv",
        "GR 8h": "../../results/geometric_biology/scLDM_GR_08_3D/gene_module_eval.csv",
        "GR 18h": "../../results/geometric_biology/scLDM_GR_18_3D/gene_module_eval.csv",
    },
}

display_labels = {
    "GR_direct_GRE_core": "GR Direct GRE",
    "GR_secondary_TF_core": "GR Late TF",
    "Glycolysis_core": "Glycolysis",
    "TCA_cycle_core": "TCA Cycle",
}

desired_module_order = [
    "GR_direct_GRE_core",
    "GR_secondary_TF_core",
    "Glycolysis_core",
    "TCA_cycle_core",
]

# -------------------------
# A priori expectation panel
# Higher = expected tighter / more coordinated
# These are not q-values
# -------------------------
a_priori_expectation = {
    "GR_direct_GRE_core": {
        "Control": 0.20,
        "GR 1h": 0.60,
        "GR 2h": 0.75,
        "GR 4h": 0.88,
        "GR 8h": 0.95,
        "GR 18h": 0.95,
    },
    "GR_secondary_TF_core": {
        "Control": 0.20,
        "GR 1h": 0.25,
        "GR 2h": 0.35,
        "GR 4h": 0.50,
        "GR 8h": 0.70,
        "GR 18h": 0.75,
    },
    "Glycolysis_core": {
        "Control": 0.90,
        "GR 1h": 0.90,
        "GR 2h": 0.90,
        "GR 4h": 0.90,
        "GR 8h": 0.90,
        "GR 18h": 0.90,
    },
    "TCA_cycle_core": {
        "Control": 0.85,
        "GR 1h": 0.85,
        "GR 2h": 0.85,
        "GR 4h": 0.85,
        "GR 8h": 0.85,
        "GR 18h": 0.85,
    },
}

def expectation_label(x):
    if x >= 0.85:
        return "high"
    if x >= 0.55:
        return "mid"
    if x >= 0.30:
        return "low"
    return "base"

def stars(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""

def to_neglog10(series_or_df):
    tiny = np.finfo(float).tiny
    return -np.log10(pd.to_numeric(series_or_df, errors="coerce").clip(lower=tiny))

def load_and_prepare(paths_by_condition, desired_module_order=None):
    parts = []
    for cond, path in paths_by_condition.items():
        df = pd.read_csv(path)

        tmp = df[["module", "qval_fdr_bh"]].copy()
        tmp["module"] = tmp["module"].astype(str).str.strip()
        tmp = tmp.rename(columns={"qval_fdr_bh": cond})
        parts.append(tmp)

    merged = reduce(lambda l, r: l.merge(r, on="module", how="inner"), parts)
    merged = merged[merged["module"] != "Immunoproteasome"].set_index("module")

    if desired_module_order is not None:
        missing = [m for m in desired_module_order if m not in merged.index]
        if missing:
            print(f"Missing modules (not in all conditions): {missing}")
        merged = merged.loc[[m for m in desired_module_order if m in merged.index]]

    qvals_raw = merged.apply(pd.to_numeric, errors="coerce")
    qvals_neglog10 = qvals_raw.apply(to_neglog10)
    annot = qvals_raw.map(stars)

    return qvals_neglog10, annot

def build_expectation_df(a_priori_dict, desired_module_order):
    exp_df = pd.DataFrame.from_dict(a_priori_dict, orient="index")
    exp_df.index.name = "module"

    exp_df = exp_df.loc[[m for m in desired_module_order if m in exp_df.index]]

    desired_cols = ["Control", "GR 1h", "GR 2h", "GR 4h", "GR 8h", "GR 18h"]
    exp_df = exp_df[desired_cols]

    annot = exp_df.map(expectation_label)
    return exp_df.astype(float), annot

# -------------------------
# Prepare data
# -------------------------
prepared = {}

exp_qvals, exp_annot = build_expectation_df(a_priori_expectation, desired_module_order)
prepared["A priori"] = {
    "qvals": exp_qvals,
    "annot": exp_annot,
    "kind": "expectation",
}

for method, paths in datasets.items():
    qvals, annot = load_and_prepare(paths, desired_module_order=desired_module_order)
    prepared[method] = {
        "qvals": qvals,
        "annot": annot,
        "kind": "qval",
    }

# Global scale for all -log10(q) panels
qval_max = max(
    float(np.nanmax(data["qvals"].to_numpy(dtype=float)))
    for data in prepared.values()
    if data["kind"] == "qval"
)
qval_max = max(qval_max, 1.0)

# -------------------------
# Plot settings
# -------------------------
k_heatmaps_per_row = 3
n_panels = len(prepared)
ncols = min(k_heatmaps_per_row, n_panels)
nrows = math.ceil(n_panels / ncols)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(FIG_W, FIG_H),
    squeeze=False
)
axes = axes.ravel()

fig.subplots_adjust(
    left=0.12,
    right=0.88,
    bottom=0.12,
    top=0.90,
    wspace=0.15,
    hspace=0.30
)

# colorbar axis
qval_cbar_ax = fig.add_axes([0.915, 0.20, 0.012, 0.60])

qval_norm = Normalize(vmin=0.0, vmax=qval_max)
qval_cmap = "Blues"

exp_norm = Normalize(vmin=0.0, vmax=1.0)
exp_cmap = "Blues"

# -------------------------
# Draw panels
# -------------------------
for i, (panel_name, data) in enumerate(prepared.items()):
    ax = axes[i]
    kind = data["kind"]

    if kind == "expectation":
        sns.heatmap(
            data["qvals"],
            ax=ax,
            cmap=exp_cmap,
            norm=exp_norm,
            cbar=False,
            linewidths=0.4,
            linecolor="white",
            square=False,
        )
    else:
        sns.heatmap(
            data["qvals"],
            ax=ax,
            cmap=qval_cmap,
            norm=qval_norm,
            annot=data["annot"],
            annot_kws={"fontsize": ANNOT_FONTSIZE},
            fmt="",
            cbar=False,
            linewidths=0.4,
            linecolor="white",
            square=False,
        )

    ax.set_title(panel_name, fontsize=TITLE_FONTSIZE, pad=4)
    ax.set_title("")
    ax.set_xlabel("")
    ax.set_ylabel("")

    ax.set_xticklabels(
        data["qvals"].columns,
        rotation=0,
        ha="center",
        fontsize=TICK_FONTSIZE
    )
    ax.tick_params(axis="x", length=0, pad=1)

    if i % ncols == 0:
        ylabels = [display_labels.get(m, m) for m in data["qvals"].index]
        ax.set_yticklabels(ylabels, rotation=0, fontsize=TICK_FONTSIZE)
        ax.tick_params(axis="y", length=0, pad=1)
    else:
        ax.set_yticklabels([])
        ax.tick_params(axis="y", length=0)

# Hide any unused axes
for j in range(n_panels, len(axes)):
    axes[j].set_visible(False)

# -------------------------
# Standalone colorbar
# -------------------------
sm = ScalarMappable(norm=qval_norm, cmap=qval_cmap)
sm.set_array([])

cbar = fig.colorbar(sm, cax=qval_cbar_ax)
cbar.set_label(r"$-\log_{10}(q)$", fontsize=CBAR_LABEL_FONTSIZE, labelpad=4)
cbar.ax.tick_params(labelsize=CBAR_TICK_FONTSIZE, length=2, pad=1)
cbar.ax.yaxis.set_major_locator(MaxNLocator(5))

# -------------------------
# Save
# -------------------------
save_path = "output/fig_6/gr_heatmap.svg"
fig.savefig(save_path, bbox_inches="tight")
plt.show()

## 2D Native space

In [ ]:
import anndata as ad

def align_external_embedding_to_adata(
    adata: ad.AnnData,
    gene_embeddings: np.ndarray,
    embedding_gene_names: np.ndarray,
) -> tuple[ad.AnnData, np.ndarray]:
    gene_embeddings = np.asarray(gene_embeddings)
    embedding_gene_names = np.asarray(embedding_gene_names).astype(str)

    if gene_embeddings.shape[0] != embedding_gene_names.shape[0]:
        raise ValueError(
            "Embedding rows must match number of genes in --embedding_gene_names_npy."
        )

    if len(pd.unique(embedding_gene_names)) != embedding_gene_names.shape[0]:
        raise ValueError("Duplicate gene names found in --embedding_gene_names_npy.")

    adata_gene_names = adata.var_names.to_numpy().astype(str)
    adata_gene_index = pd.Index(adata_gene_names)
    embedding_gene_index = pd.Index(embedding_gene_names)
    common_gene_names = adata_gene_index.intersection(embedding_gene_index, sort=False)

    if common_gene_names.empty:
        raise ValueError("No overlapping genes between adata.var_names and embedding gene names.")

    if common_gene_names.shape[0] < 2:
        raise ValueError(
            "Need at least two overlapping genes between adata and embedding for evaluation."
        )

    embedding_pos = pd.Series(np.arange(embedding_gene_names.shape[0]), index=embedding_gene_index)
    embedding_idx = embedding_pos.loc[common_gene_names].to_numpy(dtype=int)
    adata = adata[:, common_gene_names.to_numpy()].copy()
    gene_embeddings = gene_embeddings[embedding_idx]

    if gene_embeddings.shape[0] != adata.n_vars:
        raise ValueError("Failed to align embedding rows with adata genes.")

    return adata, gene_embeddings

In [ ]:
import numpy as np
import anndata as ad

adata_00 = ad.read_h5ad('../../data/GR_00.h5ad')

z_cells_scLDM_2D_00 = np.load("../../results/geometric_biology/scLDM_GR_00_2D/scLDM_cell_latent.npy") 
z_genes_scLDM_2D_00 = np.load("../../results/geometric_biology/scLDM_GR_00_2D/scLDM_gene_latent.npy") 
genes_names_scLDM_2D_00 = np.load("../../results/geometric_biology/scLDM_GR_00_2D/scLDM_gene_names.npy") 

adata_00, z_genes_scLDM_2D_00 = align_external_embedding_to_adata(adata_00, z_genes_scLDM_2D_00, genes_names_scLDM_2D_00)

adata_00.obsm["scLDM_2D"] = z_cells_scLDM_2D_00
adata_00.varm["scLDM_2D"] = z_genes_scLDM_2D_00


adata_18 = ad.read_h5ad('../../data/GR_18.h5ad')

z_cells_scLDM_2D_18 = np.load("../../results/geometric_biology/scLDM_GR_18_2D/scLDM_cell_latent.npy") 
z_genes_scLDM_2D_18 = np.load("../../results/geometric_biology/scLDM_GR_18_2D/scLDM_gene_latent.npy") 
genes_names_scLDM_2D_18 = np.load("../../results/geometric_biology/scLDM_GR_18_2D/scLDM_gene_names.npy") 

adata_18, z_genes_scLDM_2D_18 = align_external_embedding_to_adata(adata_18, z_genes_scLDM_2D_18, genes_names_scLDM_2D_18)

adata_18.obsm["scLDM_2D"] = z_cells_scLDM_2D_18
adata_18.varm["scLDM_2D"] = z_genes_scLDM_2D_18


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scanpy.plotting import palettes as scpal
import pandas as pd
from pathlib import Path
import numpy as np
from itertools import permutations

# =========================================================
# Plot settings
# =========================================================
MM_TO_INCH = 1 / 25.4
FIG_W_MM = 50
FIG_H_MM = 50
FIG_W = FIG_W_MM * MM_TO_INCH
FIG_H = FIG_H_MM * MM_TO_INCH

mpl.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 600,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 5.5,
    "axes.titlesize": 6.0,
    "axes.labelsize": 6.0,
    "xtick.labelsize": 5.0,
    "ytick.labelsize": 5.0,
    "legend.fontsize": 5.0,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


def build_palette_local(series):
    categories = list(pd.Categorical(series).categories)
    n_cat = len(categories)

    if n_cat <= 20:
        colors = scpal.default_20[:n_cat]
    elif n_cat <= 28:
        colors = scpal.default_28[:n_cat]
    elif n_cat <= 102:
        colors = scpal.default_102[:n_cat]
    else:
        cmap = plt.get_cmap("gist_ncar", n_cat)
        colors = [to_hex(cmap(i)) for i in range(n_cat)]

    return categories, dict(zip(categories, colors))


def tsp_cycle_order(points, exact_threshold=9):
    """
    Return the ordering of points giving the shortest CLOSED TSP cycle.

    For small n, solve exactly by brute force while fixing the first node
    to avoid equivalent rotations.
    For larger n, fall back to a nearest-neighbor heuristic.
    """
    points = np.asarray(points, dtype=float)
    n = len(points)

    if n <= 1:
        return np.arange(n, dtype=int)

    # Pairwise Euclidean distance matrix
    D = np.linalg.norm(points[:, None, :] - points[None, :, :], axis=2)

    # Exact solution for small number of points
    if n <= exact_threshold:
        best_order = None
        best_len = np.inf

        # Fix first node at 0 to avoid cyclic duplicates
        for perm_rest in permutations(range(1, n)):
            perm = np.array((0,) + perm_rest, dtype=int)
            cycle_len = D[perm[:-1], perm[1:]].sum() + D[perm[-1], perm[0]]
            if cycle_len < best_len:
                best_len = cycle_len
                best_order = perm

        return best_order

    # Heuristic fallback for larger n: nearest-neighbor, best start
    best_order = None
    best_len = np.inf

    for start in range(n):
        unvisited = set(range(n))
        unvisited.remove(start)
        order = [start]

        while unvisited:
            last = order[-1]
            nxt = min(unvisited, key=lambda j: D[last, j])
            order.append(nxt)
            unvisited.remove(nxt)

        order = np.asarray(order, dtype=int)
        cycle_len = D[order[:-1], order[1:]].sum() + D[order[-1], order[0]]

        if cycle_len < best_len:
            best_len = cycle_len
            best_order = order

    return best_order


# =========================================================
# User settings
# =========================================================
label = "time_hr"  # one plot per dataset
gene_name_col = None
highlight_genes = ["FKBP5", "SGK1", "DUSP1", "TSC22D3", "KLF9", "PER1", "ZFP36"]

highlight_gene_color = "#7A1E1E"
highlight_gene_size = 2.0
highlight_gene_alpha = 1.0
annotate_highlight_genes = False
show_highlight_legend = False

# TSP path settings
draw_highlight_path = True
highlight_path_color = "#1B8D3C"
highlight_path_lw = 0.5
highlight_path_alpha = 1.0

outdir = Path("output/fig_6")
outdir.mkdir(parents=True, exist_ok=True)

datasets = {
    "00": adata_00,
    "18": adata_18,
}

# =========================================================
# Precompute shared limits across both datasets
# =========================================================
all_x = []
all_y = []

for _, ad in datasets.items():
    Zc = np.asarray(ad.obsm["scLDM_2D"], dtype=float)
    Zg = np.asarray(ad.varm["scLDM_2D"], dtype=float)

    if Zc.shape[1] != 2 or Zg.shape[1] != 2:
        raise ValueError(f"Expected 2D embeddings; got cells={Zc.shape}, genes={Zg.shape}")

    all_x.append(np.concatenate([Zc[:, 0], Zg[:, 0]]))
    all_y.append(np.concatenate([Zc[:, 1], Zg[:, 1]]))

all_x = np.concatenate(all_x)
all_y = np.concatenate(all_y)

xpad = 0.03 * (all_x.max() - all_x.min() if all_x.max() > all_x.min() else 1.0)
ypad = 0.03 * (all_y.max() - all_y.min() if all_y.max() > all_y.min() else 1.0)
xlim = (all_x.min() - xpad, all_x.max() + xpad)
ylim = (all_y.min() - ypad, all_y.max() + ypad)

# Small text offset for labels
x_text_offset = 0.002 * (xlim[1] - xlim[0])
y_text_offset = 0.002 * (ylim[1] - ylim[0])

# =========================================================
# Plot function
# =========================================================
def plot_one_adata(ad, tag):
    if label not in ad.obs.columns:
        raise ValueError(f"'{label}' not found in adata_{tag}.obs")

    Z_cells_2d = np.asarray(ad.obsm["scLDM_2D"], dtype=float)
    Z_genes_2d = np.asarray(ad.varm["scLDM_2D"], dtype=float)

    if Z_cells_2d.shape[1] != 2 or Z_genes_2d.shape[1] != 2:
        raise ValueError(
            f"Expected 2D embeddings in adata_{tag}; got cells={Z_cells_2d.shape}, genes={Z_genes_2d.shape}"
        )

    if gene_name_col is None:
        gene_names = ad.var_names.astype(str).to_numpy()
    else:
        if gene_name_col not in ad.var.columns:
            raise ValueError(f"'{gene_name_col}' not found in adata_{tag}.var")
        gene_names = ad.var[gene_name_col].astype(str).to_numpy()

    gene_names_upper = pd.Series(gene_names).str.upper().to_numpy()
    highlight_set = {g.upper() for g in highlight_genes}
    highlight_mask = np.isin(gene_names_upper, list(highlight_set))

    matched = sorted(set(gene_names[highlight_mask]))
    missing = sorted(highlight_set - set(gene_names_upper))
    print(f"[adata_{tag}] matched ({len(matched)}): {matched}")
    if missing:
        print(f"[adata_{tag}] missing ({len(missing)}): {missing}")

    values = ad.obs[label].astype(str)

    if "COMMON_CATEGORIES" in globals() and "COMMON_PALETTES" in globals() and label in COMMON_CATEGORIES:
        categories = COMMON_CATEGORIES[label]
        palette = COMMON_PALETTES[label]
    else:
        categories, palette = build_palette_local(values)

    values_np = values.to_numpy()

    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))

    # -----------------------------------------------------
    # Cells
    # -----------------------------------------------------
    for ct in categories:
        idx = values_np == ct
        if np.any(idx):
            ax.scatter(
                Z_cells_2d[idx, 0],
                Z_cells_2d[idx, 1],
                s=0.4,
                c=[palette[ct]],
                alpha=0.95,
                linewidths=0,
                rasterized=True,
                zorder=2,
            )

    # -----------------------------------------------------
    # Genes: background
    # -----------------------------------------------------
    bg_mask = ~highlight_mask
    ax.scatter(
        Z_genes_2d[bg_mask, 0],
        Z_genes_2d[bg_mask, 1],
        s=0.3,
        c="#8C8C8C",
        alpha=0.35,
        linewidths=0,
        rasterized=True,
        zorder=1,
    )

    # -----------------------------------------------------
    # Highlighted genes + closed TSP cycle
    # -----------------------------------------------------
    if np.any(highlight_mask):
        pts = Z_genes_2d[highlight_mask]
        names = gene_names[highlight_mask]

        # Draw closed TSP cycle behind points
        if draw_highlight_path and pts.shape[0] >= 2:
            order = tsp_cycle_order(pts)
            pts_ordered = pts[order]
            names_ordered = names[order]

            # Close the cycle by appending the first point
            pts_closed = np.vstack([pts_ordered, pts_ordered[0]])
            names_closed = list(names_ordered) + [names_ordered[0]]

            ax.plot(
                pts_closed[:, 0],
                pts_closed[:, 1],
                color=highlight_path_color,
                linewidth=highlight_path_lw,
                alpha=highlight_path_alpha,
                zorder=4,
            )

            print(f"[adata_{tag}] TSP cycle order: {names_closed}")

        # Draw highlighted points on top
        ax.scatter(
            pts[:, 0],
            pts[:, 1],
            s=highlight_gene_size,
            c=highlight_gene_color,
            alpha=highlight_gene_alpha,
            linewidths=0.25,
            edgecolors="black",
            zorder=5,
        )

        # Labels
        if annotate_highlight_genes:
            for x, y, g in zip(pts[:, 0], pts[:, 1], names):
                ax.text(
                    x + x_text_offset,
                    y + y_text_offset,
                    g,
                    fontsize=2,
                    ha="left",
                    va="bottom",
                    color="black",
                    zorder=6,
                )

    # -----------------------------------------------------
    # Styling
    # -----------------------------------------------------
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title("")

    for spine in ax.spines.values():
        spine.set_visible(False)

    svg_path = outdir / f"kang_scLDM_2d_{label}_adata_{tag}.svg"
    fig.savefig(svg_path, bbox_inches="tight", pad_inches=0, transparent=False)
    plt.close(fig)

    print(f"Saved: {svg_path}")


# =========================================================
# Run for adata_00 and adata_18
# =========================================================
plot_one_adata(adata_00, "00")
plot_one_adata(adata_18, "18")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def mean_pairwise_distance(pts):
    # pts shape: (k, 2)
    diffs = pts[:, None, :] - pts[None, :, :]
    dmat = np.sqrt(np.sum(diffs**2, axis=2))
    iu = np.triu_indices(dmat.shape[0], k=1)
    return float(dmat[iu].mean())

def permutation_test_gene_set_distance(
    Z_genes_2d,
    gene_names,
    gene_list,
    n_perm=10_000,
    seed=42,
    require_all_genes=True,
):
    Z = np.asarray(Z_genes_2d, dtype=float)
    gene_names = np.asarray(gene_names, dtype=str)
    gene_names_upper = pd.Series(gene_names).str.upper().to_numpy()

    target = [g.upper() for g in gene_list]
    target_set = set(target)

    mask_obs = np.isin(gene_names_upper, target)
    found = gene_names[mask_obs].tolist()
    missing = sorted(target_set - set(gene_names_upper))

    if require_all_genes and missing:
        raise ValueError(f"Missing genes in embedding: {missing}")

    k = len(target) if require_all_genes else int(mask_obs.sum())
    if k < 2:
        raise ValueError("Need at least 2 genes for pairwise distance.")

    # observed
    if require_all_genes:
        obs_pts = Z[mask_obs]
    else:
        obs_pts = Z[mask_obs]
    observed = mean_pairwise_distance(obs_pts)

    # permutations
    rng = np.random.default_rng(seed)
    n = Z.shape[0]
    null = np.empty(n_perm, dtype=float)
    for i in range(n_perm):
        idx = rng.choice(n, size=k, replace=False)
        null[i] = mean_pairwise_distance(Z[idx])

    # p-values (if small distance means tighter clustering, use p_lower)
    p_lower = (np.sum(null <= observed) + 1) / (n_perm + 1)
    p_upper = (np.sum(null >= observed) + 1) / (n_perm + 1)
    z = (observed - null.mean()) / null.std(ddof=1)

    return {
        "observed": observed,
        "null": null,
        "p_lower": p_lower,
        "p_upper": p_upper,
        "zscore": z,
        "found": found,
        "missing": missing,
        "k": k,
    }

highlight_genes = ["FKBP5","SGK1","DUSP1","TSC22D3","KLF9","PER1","ZFP36"]

for tag, ad, gnames in [
    ("adata_00", adata_00, genes_names_scLDM_2D_00),
    ("adata_18", adata_18, genes_names_scLDM_2D_18),
]:
    res = permutation_test_gene_set_distance(
        Z_genes_2d=ad.varm["scLDM_2D"],
        gene_names=gnames,
        gene_list=highlight_genes,
        n_perm=10_000,
        seed=42,
        require_all_genes=True,
    )

    
    from scipy.spatial.distance import pdist, cdist

    Z = np.asarray(ad.varm["scLDM_2D"], dtype=float)
    gene_names = np.asarray(ad.var_names, dtype=str)
    module_mask = np.isin(
        np.char.upper(gene_names),
        [g.upper() for g in highlight_genes],
    )

    within = pdist(Z[module_mask], metric="euclidean").mean()
    versus_rest = cdist(
        Z[module_mask], Z[~module_mask], metric="euclidean"
    ).mean()

    compactness_ratio = within / versus_rest

    print(f"\n[{tag}]")
    print(f"Observed mean pairwise distance: {res['observed']:.4f}")
    print(f"Null mean ± sd: {res['null'].mean():.4f} ± {res['null'].std(ddof=1):.4f}")
    print(f"p_lower (more clustered): {res['p_lower']:.5f}")
    print(f"p_upper (more dispersed): {res['p_upper']:.5f}")
    print(f"z-score: {res['zscore']:.3f}")
    print(f"Found ({len(res['found'])}): {res['found']}")
    print(f"Compactness Ratio: {compactness_ratio:.3f}")

    # optional quick histogram
    plt.figure(figsize=(3.2, 2.2))
    plt.hist(res["null"], bins=50, color="0.75", edgecolor="none")
    plt.axvline(res["observed"], color="#d62728", lw=1.5)
    plt.title(tag)
    plt.xlabel("Mean pairwise distance")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()


## TCR Heatmap - Shared Space

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import MaxNLocator

# =========================================================
# Figure dimensions
# =========================================================
MM_TO_INCH = 1 / 25.4

# Final submission size
FIG_W_MM = 180   # double-column width
FIG_H_MM = 30    # compact heatmap height; increase slightly if labels feel cramped

FIG_W = FIG_W_MM * MM_TO_INCH
FIG_H = FIG_H_MM * MM_TO_INCH

# Font sizes
BASE_FONTSIZE = 5.5
TITLE_FONTSIZE = 6.0
GROUP_LABEL_FONTSIZE = 6.0
TICK_FONTSIZE = 5.0
YTICK_FONTSIZE = 5.0
ANNOT_FONTSIZE = 5.0
CBAR_LABEL_FONTSIZE = 6.0
CBAR_TICK_FONTSIZE = 5.0

sns.set_style("white")
plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": BASE_FONTSIZE,
    "axes.titlesize": TITLE_FONTSIZE,
    "axes.labelsize": BASE_FONTSIZE,
    "xtick.labelsize": TICK_FONTSIZE,
    "ytick.labelsize": YTICK_FONTSIZE,
    "pdf.fonttype": 42,   # editable text in Illustrator
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

# =========================================================
# Paths
# =========================================================
scldm_csv = "../../results/geometric_biology/scLDM_TCR/gene_module_eval.csv"
simba_csv = "../../results/geometric_biology/SIMBA_TCR/gene_module_eval.csv"

save_path = "output/fig_6/tf_heatmap.svg"

# =========================================================
# Custom grouped TF sets for Figure 6
# =========================================================
figure6_tcr_tf_demo_sets_grouped = {
    "Immediate TCR transcription": [
        "NFATC2", "NFATC1",
        "RELA", "REL", "NFKB1", "NFKB2",
        "EGR1", "ELK1",
        "JUN", "JUND",
        "ATF1", "ATF2", "ATF3"
    ],
    "Cytokine response (STAT/IRF)": [
        "STAT1", "STAT2", "STAT3", "STAT4", "STAT5A", "STAT5B",
        "IRF1", "IRF2", "IRF9"
    ],
    "Anabolic metabolism": [
        "MYC", "MAX", "HIF1A", "SREBF2"
    ],
}

# =========================================================
# Settings
# =========================================================
show_only_union_hits = False   # True = keep only TFs with min(q_scLDM, q_SIMBA) < q_thresh
q_thresh = 0.10
cmap = "Blues"


# If gene symbols are not in obs_names, set this to the correct obs column name.
gene_symbol_col = None

highlight_gene_size = 5.0
highlight_gene_alpha = 0.95
module_colors = {
    "Immediate TCR transcription": "#d62728",
    "Cytokine response (STAT/IRF)": "#1f77b4",
    "Anabolic metabolism": "#2ca02c",
}


# =========================================================
# Helpers
# =========================================================
def clean_tf(x):
    return pd.Index(x).astype(str).str.strip().str.upper()

def to_neglog10(x):
    tiny = np.finfo(float).tiny
    if isinstance(x, pd.DataFrame):
        x = x.apply(pd.to_numeric, errors="coerce")
    else:
        x = pd.to_numeric(x, errors="coerce")
    return -np.log10(x.clip(lower=tiny))

def stars(q):
    if pd.isna(q):
        return ""
    if q < 0.001:
        return "***"
    if q < 0.01:
        return "**"
    if q < 0.05:
        return "*"
    return ""

# =========================================================
# Load results
# =========================================================
scldm = pd.read_csv(scldm_csv)
simba = pd.read_csv(simba_csv)

scldm["TF"] = clean_tf(scldm["module"])
simba["TF"] = clean_tf(simba["module"])

scldm_meta = (
    scldm[["TF", "qval_fdr_bh"]]
    .drop_duplicates("TF")
    .rename(columns={"qval_fdr_bh": "scLDM q-value"})
)

simba_meta = (
    simba[["TF", "qval_fdr_bh"]]
    .drop_duplicates("TF")
    .rename(columns={"qval_fdr_bh": "SIMBA q-value"})
)

meta = scldm_meta.merge(simba_meta, on="TF", how="outer")
meta["SCENE q-value"] = pd.to_numeric(meta["scLDM q-value"], errors="coerce")
meta["SIMBA q-value"] = pd.to_numeric(meta["SIMBA q-value"], errors="coerce")
meta["min_q"] = meta[["SCENE q-value", "SIMBA q-value"]].min(axis=1, skipna=True)

# =========================================================
# Build ordered blocks from grouped TF sets
# =========================================================
blocks = []
missing_tfs = []

for group_name, tf_list in figure6_tcr_tf_demo_sets_grouped.items():
    tf_list = [tf.upper() for tf in tf_list]

    sub = meta[meta["TF"].isin(tf_list)].copy()
    sub["TF"] = pd.Categorical(sub["TF"], categories=tf_list, ordered=True)
    sub = sub.sort_values("TF").reset_index(drop=True)

    found = set(sub["TF"].astype(str))
    missing = [tf for tf in tf_list if tf not in found]
    if missing:
        missing_tfs.extend([(group_name, tf) for tf in missing])

    if show_only_union_hits:
        sub = sub[sub["min_q"] < q_thresh].copy()

    if not sub.empty:
        blocks.append((group_name, sub))

if len(blocks) == 0:
    raise ValueError("No TFs left to plot after filtering.")

if missing_tfs:
    print("TFs not found in scLDM/SIMBA files:")
    for group_name, tf in missing_tfs:
        print(f"  {group_name}: {tf}")

# =========================================================
# Shared scale across all TF blocks
# =========================================================
q_max = max(
    float(np.nanmax(to_neglog10(sub[["SCENE q-value", "SIMBA q-value"]]).to_numpy(dtype=float)))
    for _, sub in blocks
)
q_max = max(q_max, 1.0)
norm = Normalize(vmin=0.0, vmax=q_max)

# =========================================================
# Layout
# =========================================================
# Use a small blank spacer axis between TF groups
gap_ratio = 0.6
width_ratios = []
for i, (_, sub) in enumerate(blocks):
    width_ratios.append(len(sub))
    if i < len(blocks) - 1:
        width_ratios.append(gap_ratio)

fig, axes = plt.subplots(
    1,
    len(width_ratios),
    figsize=(FIG_W, FIG_H),
    gridspec_kw={"width_ratios": width_ratios, "wspace": 0.04},
)

if not isinstance(axes, np.ndarray):
    axes = np.array([axes])

# Leave room for rotated TF labels and colorbar
fig.subplots_adjust(
    left=0.07,
    right=0.89,
    bottom=0.34,
    top=0.82,
)

plot_axes = []
for i, ax in enumerate(axes):
    if i % 2 == 1:
        ax.axis("off")
    else:
        plot_axes.append(ax)

# =========================================================
# Draw grouped heatmaps
# =========================================================
for i, (ax, (group_name, sub)) in enumerate(zip(plot_axes, blocks)):
    q_df = sub[["SCENE q-value", "SIMBA q-value"]].T
    q_df.index = ["SCENE", "SIMBA"]

    display_df = to_neglog10(q_df)
    # annot_df = q_df.applymap(stars)  # Optional significance stars

    sns.heatmap(
        display_df,
        ax=ax,
        cmap=cmap,
        norm=norm,
        linewidths=0.4,
        linecolor="white",
        cbar=False,
        # annot=annot_df,
        fmt="",
        annot_kws={"fontsize": ANNOT_FONTSIZE},
        xticklabels=sub["TF"].astype(str).tolist(),
        yticklabels=(["SCENE", "SIMBA"] if i == 0 else False),
    )

    ax.set_title("")
    ax.set_xlabel("")
    ax.set_ylabel("")

    ax.tick_params(axis="x", rotation=45, labelsize=TICK_FONTSIZE, length=0, pad=1)
    ax.tick_params(axis="y", rotation=0, labelsize=YTICK_FONTSIZE, length=0, pad=3)


# =========================================================
# Standalone colorbar
# =========================================================
# [left, bottom, width, height]
cbar_ax = fig.add_axes([0.91, 0.34, 0.010, 0.40])

sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label(r"$-\log_{10}(q)$", fontsize=CBAR_LABEL_FONTSIZE, labelpad=4)
cbar.ax.tick_params(labelsize=CBAR_TICK_FONTSIZE, length=2, pad=1)
cbar.ax.yaxis.set_major_locator(MaxNLocator(4))

# =========================================================
# Save / show
# =========================================================
os.makedirs(os.path.dirname(save_path), exist_ok=True)
fig.savefig(save_path, bbox_inches="tight", dpi=450)
plt.show()

print(f"Saved to: {save_path}")

## Vertical significance jitter

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# =========================================================
# Plot settings
# =========================================================
MM_TO_INCH = 1 / 25.4

# Panel size
FIG_W_MM = 50   # widened a bit to make room for legend on the right
FIG_H_MM = 40
FIG_W = FIG_W_MM * MM_TO_INCH
FIG_H = FIG_H_MM * MM_TO_INCH

Q_THRESH = 0.05
THRESH_Y = -np.log10(Q_THRESH)

mpl.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 5.5,
    "axes.labelsize": 6.0,
    "xtick.labelsize": 5.0,
    "ytick.labelsize": 5.0,
    "legend.fontsize": 5.0,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

# =========================================================
# Helper functions
# =========================================================
def to_neglog10(x):
    x = pd.to_numeric(x, errors="coerce").astype(float)
    tiny = np.finfo(float).tiny
    return -np.log10(np.clip(x, tiny, None))

def jitter_qvals_nature(
    ax,
    modules,
    left,
    right=None,
    left_label="scLDM",
    right_label="SIMBA",
    q_thresh=0.05,
    seed=42,
):
    modules = np.asarray(modules, dtype=object)
    left = np.asarray(left, dtype=float)
    left_plot = to_neglog10(left)

    thresh_line = -np.log10(q_thresh)
    rng = np.random.default_rng(seed)

    # Tighter jitter for small publication panels
    x0 = rng.normal(0.0, 0.1, size=left.shape[0])

    # Plot colors
    colors = {
        "both": "#009E73",   # bluish green
        "left": "#D55E00",   # vermillion
        "right": "#0072B2",  # blue
        "other": "#B3B3B3",  # gray
    }

    point_size = 3

    if right is None:
        left_sig = left < q_thresh

        ax.scatter(
            x0[~left_sig],
            left_plot[~left_sig],
            s=point_size,
            c=colors["other"],
            alpha=0.8,
            linewidths=0,
            rasterized=True,
        )
        ax.scatter(
            x0[left_sig],
            left_plot[left_sig],
            s=point_size,
            c=colors["left"],
            alpha=0.9,
            linewidths=0,
            rasterized=True,
        )

        ax.axhline(thresh_line, color="black", lw=0.6, ls="--")
        ax.set_xlim(-0.26, 0.26)
        ax.set_xticks([0])
        ax.set_xticklabels([left_label])
        ax.set_ylabel(r"$-\log_{10}(q)$")

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.legend(
            handles=[
                Line2D(
                    [0], [0],
                    marker="o",
                    color="none",
                    label=f"{left_label} q < {q_thresh}",
                    markerfacecolor=colors["left"],
                    markeredgewidth=0,
                    markersize=3.6,
                ),
                Line2D(
                    [0], [0],
                    marker="o",
                    color="none",
                    label="Other",
                    markerfacecolor=colors["other"],
                    markeredgewidth=0,
                    markersize=3.6,
                ),
            ],
            frameon=False,
            loc="upper left",
            bbox_to_anchor=(1.02, 1.0),
            handletextpad=0.4,
            borderaxespad=0.0,
            labelspacing=0.3,
        )
        return

    right = np.asarray(right, dtype=float)
    right_plot = to_neglog10(right)
    x1 = rng.normal(1.0, 0.1, size=right.shape[0])

    left_sig = left < q_thresh
    right_sig = right < q_thresh

    both_sig = left_sig & right_sig
    left_only = left_sig & ~right_sig
    right_only = ~left_sig & right_sig
    neither = ~(both_sig | left_only | right_only)

    for mask, c in [
        (neither, colors["other"]),
        (both_sig, colors["both"]),
        (left_only, colors["left"]),
        (right_only, colors["right"]),
    ]:
        ax.scatter(
            x0[mask],
            left_plot[mask],
            s=point_size,
            c=c,
            alpha=0.85,
            linewidths=0,
            rasterized=True,
        )
        ax.scatter(
            x1[mask],
            right_plot[mask],
            s=point_size,
            c=c,
            alpha=0.85,
            linewidths=0,
            rasterized=True,
        )

    ax.axhline(thresh_line, color="black", lw=0.6, ls="--")
    ax.set_xlim(-0.35, 1.35)
    ax.set_xticks([0, 1])
    ax.set_xticklabels([left_label, right_label])
    ax.set_ylabel(r"$-\log_{10}(q)$")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.legend(
        handles=[
            Line2D(
                [0], [0],
                marker="o",
                color="none",
                label=f"Both q < {q_thresh}",
                markerfacecolor=colors["both"],
                markeredgewidth=0,
                markersize=3.6,
            ),
            Line2D(
                [0], [0],
                marker="o",
                color="none",
                label=f"{left_label} only",
                markerfacecolor=colors["left"],
                markeredgewidth=0,
                markersize=3.6,
            ),
            Line2D(
                [0], [0],
                marker="o",
                color="none",
                label=f"{right_label} only",
                markerfacecolor=colors["right"],
                markeredgewidth=0,
                markersize=3.6,
            ),
            Line2D(
                [0], [0],
                marker="o",
                color="none",
                label="Other",
                markerfacecolor=colors["other"],
                markeredgewidth=0,
                markersize=3.6,
            ),
        ],
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        handletextpad=0.4,
        borderaxespad=0.0,
        labelspacing=0.3,
    )

# =========================================================
# Load data
# =========================================================
sc_path = "../../results/geometric_biology/scLDM_TCR/gene_module_eval.csv"
si_path = "../../results/geometric_biology/SIMBA_TCR/gene_module_eval.csv"

sc = pd.read_csv(sc_path)[["module", "qval_fdr_bh"]].rename(columns={"qval_fdr_bh": "q_SCENE"})
si = pd.read_csv(si_path)[["module", "qval_fdr_bh"]].rename(columns={"qval_fdr_bh": "q_SIMBA"})

m = sc.merge(si, on="module", how="inner")
m["best_q"] = m[["q_SCENE", "q_SIMBA"]].min(axis=1)
m = m.sort_values("best_q", ascending=True).reset_index(drop=True)

# =========================================================
# Plot
# =========================================================
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))

jitter_qvals_nature(
    ax,
    modules=m["module"].to_numpy(),
    left=m["q_SCENE"].to_numpy(),
    right=m["q_SIMBA"].to_numpy(),
    left_label="SCENE 16D",
    right_label="SIMBA",
    q_thresh=Q_THRESH,
)

ymax = max(
    to_neglog10(m["q_SCENE"]).max(),
    to_neglog10(m["q_SIMBA"]).max(),
)
ax.set_ylim(0, ymax * 1.05)

# Leave room for legend outside the axes
fig.subplots_adjust(right=0.68)
plt.savefig("output/fig_6/qval_jitter.svg", bbox_inches="tight", dpi=450)

plt.show()